# 🖥️ Which RunPod GPU should I use?

This notebook loads **Mistral-7B-Instruct-v0.3** (~7.2 B params). The VRAM you need depends on the **precision** you pick in the selector below — choose a pod that clears the *VRAM* column:

| Precision | Weights | ~VRAM (with overhead) | Recommended RunPod GPU | ~Price / hr |
|:---|:---|:---|:---|:---|
| **4-bit** (NF4) | ~3.6 GB | ~6 GB | **RTX A5000 · 24 GB** | **\$0.27** |
| **16-bit** (bf16) | ~14.5 GB | ~16–18 GB | **RTX A5000 · 24 GB** | **\$0.27** |
| **32-bit** (fp32) | ~29 GB | ~30–32 GB | **RTX A6000 · 48 GB** (or A40 48 GB) | **\$0.49** (A40 \$0.44) |

_Prices: RunPod Community Cloud, on-demand, ~July 2026 — they change often, check [runpod.io/pricing](https://www.runpod.io/pricing). Also set the pod's **Container/Volume disk to ≥ 100 GB**._


# Install libraries

In [1]:
%pip install torch
%pip install --upgrade transformers
%pip install bitsandbytes
%pip install accelerate
%pip install peft
%pip install datasets
%pip install evaluate
%pip install trl
%pip install matplotlib
%pip install tensorboard
%pip install sentencepiece



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 55.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.4/801.4 kB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 51.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 81.8 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 ->

In [2]:
import torch
for i in range(torch.cuda.device_count()):
   print(torch.cuda.get_device_properties(i).name)


NVIDIA L4


In [3]:
from huggingface_hub import login

API_TOKEN = "HF_TOKEN_REMOVED"
login(token=API_TOKEN)


# ⚙️ Choose precision -- run ONE of the three cells below

Run **exactly one** of the 4-bit / 16-bit / 32-bit cells. It sets `PRECISION` and `MODEL_KWARGS`,
which every section below uses (model loading, the on-disk folder `SAVED_MODEL_<precision>`, and the
Hub repo names `my-base-mistral-<precision>-...`). To switch precision later, run a different cell,
then re-run the **Load the model** cell.

In [4]:
# --- 4-bit (bitsandbytes NF4) ---
import torch
from transformers import BitsAndBytesConfig

PRECISION = "4bit"
MODEL_KWARGS = dict(
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    ),
    dtype=torch.bfloat16,
    device_map="cuda",
)
print("Selected precision:", PRECISION)


Selected precision: 4bit


In [ ]:
# --- 16-bit (half precision, bfloat16) ---
import torch

PRECISION = "16bit"
MODEL_KWARGS = dict(
    dtype=torch.bfloat16,
    device_map="cuda",
)
print("Selected precision:", PRECISION)


In [ ]:
# --- 32-bit (full precision, float32) ---
import torch

PRECISION = "32bit"
MODEL_KWARGS = dict(
    dtype=torch.float32,
    device_map="cuda",
)
print("Selected precision:", PRECISION)


# Base model from HUB

**What this notebook does:** download the original `mistralai/Mistral-7B-Instruct-v0.3` from the Hugging Face Hub at the precision you selected, verify it generates a coherent answer, then persist it — save a copy to local disk **and** push a copy to your own Hub repo.

```mermaid
flowchart LR
    ORIG["🤗 Original Hub<br/>Mistral-7B-Instruct-v0.3"] -->|load| BM["base_model"]
    BM -->|test| OK["✅ generates"]
    BM -->|save| DISK["💾 disk<br/>SAVED_MODEL_&lt;prec&gt;"]
    BM -->|push| HUB["📦 Your Hub<br/>…-from-hub"]
    style ORIG fill:#2e7d32,color:#fff,stroke:#1b5e20
```


### Load the model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Silence transformers' deprecation notice about `torch_dtype` in the model's
# own config.json (upstream in the Mistral repo; harmless).
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

# Names derived from the precision you selected above -- used by every section below.
SAVED_DIR      = f"SAVED_MODEL_{PRECISION}"
REPO_FROM_HUB  = f"lukaskellerstein/my-base-mistral-{PRECISION}-from-hub"
REPO_FROM_DISK = f"lukaskellerstein/my-base-mistral-{PRECISION}-from-disk"

base_model = AutoModelForCausalLM.from_pretrained(model_name, **MODEL_KWARGS)
base_tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loaded {model_name}  |  precision={PRECISION}  |  hub repo={REPO_FROM_HUB}")


Log model and tokenizer

In [ ]:
def describe(model, tokenizer):
    """Print key facts about a loaded model + tokenizer. Reused in every section below."""
    print("--- Model ---")
    print("Type:", type(model))
    print("Architecture:", model)
    print("Vocab size:", model.config.vocab_size)
    print("Input embeddings:", model.get_input_embeddings())
    print("Output embeddings:", model.get_output_embeddings())
    print("--- Tokenizer ---")
    print("Type:", type(tokenizer))
    print("Special tokens:", tokenizer.special_tokens_map)
    print("All tokens count:", len(tokenizer))
    print("Padding side:", tokenizer.padding_side)


describe(base_model, base_tokenizer)


### Test the model

#### via Pipeline

In [ ]:
from transformers import pipeline

PROMPT = "What is AutoGen in abstract?"


def test_pipeline(model, tokenizer, prompt=PROMPT):
    """Test a model with a text-generation pipeline. Reused in every section below."""
    my_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

    # Mistral-7B-Instruct is instruction-tuned, so we pass a chat (list of messages)
    # instead of a raw string. The pipeline then auto-applies the tokenizer's chat
    # template, giving a coherent answer (a raw string would ignore the template and ramble).
    messages = [{"role": "user", "content": prompt}]
    result = my_pipeline(messages, max_new_tokens=256, return_full_text=False)
    print(result[0]["generated_text"])


test_pipeline(base_model, base_tokenizer)


#### via Model and Tokenizer

In [ ]:
def test_generate(model, tokenizer, prompt=PROMPT):
    """Test a model by manually applying the chat template + generate. Reused below."""
    # Instruction-tuned model -> build inputs from a chat via the tokenizer's chat template.
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True
    ).to("cuda")

    # Generate, then decode only the answer.
    result = model.generate(**model_inputs, max_new_tokens=256)
    print(tokenizer.decode(result[0], skip_special_tokens=True))


test_generate(base_model, base_tokenizer)


#### via Tool call (single turn)

In [ ]:
# Single-turn tool calling: pass `tools` to the chat template. An instruction-tuned
# model should answer with a tool call (function name + JSON arguments) instead of prose.
STOCK_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current price of a stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The ticker symbol for the stock, e.g. GOOG",
                    }
                },
                "required": ["ticker"],
            },
        },
    }
]


def test_tool_call(model, tokenizer, prompt, tools, max_new_tokens=256):
    """Single-turn tool calling: pass `tools`; the model should emit a tool call. Reused below."""
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, tools=tools , add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)

    # Greedy decoding -> deterministic tool call. Keep special tokens so the
    # tool-call markers (e.g. [TOOL_CALLS]) stay visible.
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_call(base_model, base_tokenizer, "What is the current price of MSFT?", STOCK_TOOLS)


#### via Conversation with tool calls

In [ ]:
def test_tool_conversation(model, tokenizer, max_new_tokens=256):
    """Multi-turn conversation that already contains an assistant tool call + a tool
    response, then a fresh user turn. Confirms the chat template round-trips the
    `tool_calls` / `tool` roles, then answers the latest turn. Reused below."""
    conversation = [
        {"role": "user", "content": "What's the weather like in Paris?"},
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "id": "call12345",
                    "type": "function",
                    "function": {
                        "name": "get_current_weather",
                        "arguments": '{"location": "Paris, France", "format": "celsius"}',
                    },
                }
            ],
        },
        {"role": "tool", "content": '{"content": 22}', "tool_call_id": "call12345"},
        {
            "role": "assistant",
            "content": "The current temperature in Paris, France is 22 degrees Celsius.",
        },
        {"role": "user", "content": "What's the weather like in San Francisco right now?"},
    ]

    weather_tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        },
                        "format": {
                            "type": "string",
                            "enum": ["celsius", "fahrenheit"],
                            "description": "The temperature unit to use. Infer this from the users location.",
                        },
                    },
                    "required": ["location", "format"],
                },
            },
        }
    ]

    model_inputs = tokenizer.apply_chat_template(
        conversation, tools=weather_tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    print("Tokenized conversation input_ids:")
    print(model_inputs["input_ids"])

    # Answer the latest user turn (should be another get_current_weather tool call for SF).
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print("\nModel response:")
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_conversation(base_model, base_tokenizer)


### Save the model (to disk)

In [ ]:
base_model.save_pretrained(SAVED_DIR)

base_tokenizer.save_pretrained(SAVED_DIR)


### Push the model (to HUB)

In [ ]:
# push_to_hub() -> save_pretrained(save_original_format=True) can raise NotImplementedError
# in transformers 5.x, so save with save_original_format=False and upload the folder instead.
from huggingface_hub import upload_folder, create_repo

tmp = f"PUSH_TMP_{PRECISION}_from_hub"
create_repo(REPO_FROM_HUB, token=API_TOKEN, exist_ok=True)
base_model.save_pretrained(tmp, save_original_format=False)
base_tokenizer.save_pretrained(tmp)
upload_folder(repo_id=REPO_FROM_HUB, folder_path=tmp, token=API_TOKEN)
